# Deep Learning Visual Search & Image Similarity Engine — UT-Zappos50K

**Goal:** Given a user-uploaded photo of a shoe, retrieve visually similar shoes
from a catalog — even when product names/text metadata differ.

**Approach**
1. Build a catalog index (image path + category/subcategory/brand) from the folder structure.
2. Train a CNN embedding model (ResNet50 backbone + projection head) with **triplet loss**,
   using category/subcategory/brand as a weak-supervision signal for "same shoe type = similar".
3. Extract an embedding vector for every catalog image.
4. Build a fast nearest-neighbor index (FAISS, cosine similarity) over all embeddings.
5. At query time: embed the uploaded image → search the index → return top-K visually similar shoes.
6. Evaluate retrieval quality with Precision@K (category/subcategory agreement) and visualize results.

**Dataset:** [UT-Zappos50K](https://www.kaggle.com/datasets/mesevac/utzap50k) — attach it to this notebook via
`Add Input` in Kaggle. This notebook auto-detects the mounted path.


## 1. Setup & Configuration

In [ ]:
import os, sys, glob, random, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# ---- Locate the dataset root automatically (Kaggle mounts inputs under /kaggle/input/<dataset-slug>/...) ----
CANDIDATE_ROOTS = glob.glob("/kaggle/input/*")
IMAGE_DIR = None

for root in CANDIDATE_ROOTS:
    # look for a directory that contains the Boots/Sandals/Shoes/Slippers top-level categories
    for dirpath, dirnames, filenames in os.walk(root):
        names = set(dirnames)
        if {"Boots", "Sandals", "Shoes", "Slippers"} & names:
            IMAGE_DIR = dirpath
            break
    if IMAGE_DIR:
        break

# Fallback for local/dev runs outside Kaggle
if IMAGE_DIR is None:
    for guess in ["./ut-zap50k-images", "./data/ut-zap50k-images", "/kaggle/working/ut-zap50k-images"]:
        if os.path.isdir(guess):
            IMAGE_DIR = guess
            break

assert IMAGE_DIR is not None, (
    "Could not locate the ut-zap50k-images folder. "
    "Attach the UT-Zap50K dataset to this notebook (Add Input) and re-run."
)
print("Using image root:", IMAGE_DIR)

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Build the catalog index

We walk the folder tree `Category / SubCategory / Brand / *.jpg` and turn it into a
DataFrame. This gives us weak labels (category, subcategory, brand) that we'll use as a
similarity proxy for training — two shoes in the same subcategory + brand are assumed to be
visually closer than two random shoes.


In [ ]:
def build_catalog(image_dir):
    records = []
    exts = (".jpg", ".jpeg", ".png")
    for dirpath, dirnames, filenames in os.walk(image_dir):
        for fn in filenames:
            if fn.lower().endswith(exts):
                rel = os.path.relpath(os.path.join(dirpath, fn), image_dir)
                parts = rel.split(os.sep)
                # Expected shape: Category/SubCategory/Brand/file.jpg
                category    = parts[0] if len(parts) > 0 else "Unknown"
                subcategory = parts[1] if len(parts) > 1 else "Unknown"
                brand       = parts[2] if len(parts) > 2 else "Unknown"
                records.append({
                    "path": os.path.join(dirpath, fn),
                    "category": category,
                    "subcategory": subcategory,
                    "brand": brand,
                    "class_label": f"{category}/{subcategory}/{brand}",
                })
    return pd.DataFrame(records)

catalog = build_catalog(IMAGE_DIR)
print(f"Found {len(catalog):,} images across {catalog['category'].nunique()} categories, "
      f"{catalog['subcategory'].nunique()} subcategories, {catalog['brand'].nunique()} brands.")
catalog.head()


In [ ]:
# Optional: for fast experimentation, cap dataset size (comment out for full training run)
MAX_IMAGES = None  # e.g. 15000 to subsample; None = use all

if MAX_IMAGES is not None and len(catalog) > MAX_IMAGES:
    catalog = catalog.groupby("class_label", group_keys=False).apply(
        lambda x: x.sample(min(len(x), max(1, int(MAX_IMAGES * len(x) / len(catalog)))), random_state=SEED)
    ).reset_index(drop=True)
    print("Subsampled to", len(catalog), "images")

# Keep only classes with >= 2 images so triplet mining always has a positive candidate
class_counts = catalog["class_label"].value_counts()
valid_classes = class_counts[class_counts >= 2].index
catalog = catalog[catalog["class_label"].isin(valid_classes)].reset_index(drop=True)
print("Usable images after filtering singleton classes:", len(catalog))


## 3. Train / validation split (by class, so both splits see every shoe type)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    catalog, test_size=0.1, random_state=SEED,
    stratify=catalog["class_label"] if catalog["class_label"].nunique() < len(catalog) * 0.5 else None
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("Train:", len(train_df), " Val:", len(val_df))


## 4. Dataset & Transforms

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class TripletShoeDataset(Dataset):
    """
    Samples (anchor, positive, negative) triplets on the fly.
    Positive = same class_label (category/subcategory/brand) as anchor.
    Negative = different class_label.
    """
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.by_class = self.df.groupby("class_label").indices  # class_label -> array of row indices
        self.classes = list(self.by_class.keys())

    def __len__(self):
        return len(self.df)

    def _load(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img)

    def __getitem__(self, idx):
        anchor_row = self.df.iloc[idx]
        anchor_class = anchor_row["class_label"]

        pos_candidates = self.by_class[anchor_class]
        pos_idx = idx
        tries = 0
        while pos_idx == idx and tries < 10:
            pos_idx = random.choice(pos_candidates)
            tries += 1

        neg_class = random.choice(self.classes)
        while neg_class == anchor_class:
            neg_class = random.choice(self.classes)
        neg_idx = random.choice(self.by_class[neg_class])

        anchor = self._load(idx)
        positive = self._load(pos_idx)
        negative = self._load(neg_idx)
        return anchor, positive, negative


class CatalogImageDataset(Dataset):
    """Plain dataset for feature extraction / evaluation (no triplet sampling)."""
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img), idx


## 5. Embedding Model

A ResNet50 backbone (ImageNet-pretrained) with its classification head replaced by a small
projection head producing an L2-normalized 256-d embedding. We fine-tune the last residual
block plus the projection head; earlier layers stay frozen for speed and to avoid overfitting
on ~50K images.


In [ ]:
EMBED_DIM = 256

class ShoeEmbeddingNet(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, freeze_backbone_until="layer4"):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()  # remove classifier, keep pooled features
        self.backbone = backbone

        # Freeze everything up to (not including) `freeze_backbone_until`
        freeze = True
        for name, child in self.backbone.named_children():
            if name == freeze_backbone_until:
                freeze = False
            for p in child.parameters():
                p.requires_grad = not freeze

        self.projection = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, embed_dim),
        )

    def forward(self, x):
        feats = self.backbone(x)               # (B, 2048)
        emb = self.projection(feats)            # (B, embed_dim)
        emb = F.normalize(emb, p=2, dim=1)      # L2-normalize -> cosine similarity == dot product
        return emb

model = ShoeEmbeddingNet().to(DEVICE)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,}")


## 6. Training Loop (Triplet Loss)

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 8          # increase for a full training run; 8 is a reasonable Kaggle-GPU budget
LR = 1e-4
MARGIN = 0.3

train_ds = TripletShoeDataset(train_df, train_transform)
val_ds   = TripletShoeDataset(val_df, eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


In [ ]:
def run_epoch(loader, train=True):
    model.train(mode=train)
    total_loss, n_batches = 0.0, 0
    for anchor, positive, negative in loader:
        anchor, positive, negative = anchor.to(DEVICE), positive.to(DEVICE), negative.to(DEVICE)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                emb_a = model(anchor)
                emb_p = model(positive)
                emb_n = model(negative)
                loss = criterion(emb_a, emb_p, emb_n)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


history = {"train_loss": [], "val_loss": []}
best_val = float("inf")
CKPT_PATH = os.path.join(OUTPUT_DIR, "shoe_embedding_model.pt")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), CKPT_PATH)

    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} "
          f"| {time.time()-t0:.1f}s")

print("Best val loss:", best_val, "-> checkpoint saved to", CKPT_PATH)


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("triplet loss"); plt.legend(); plt.title("Training curve")
plt.show()


## 7. Extract Embeddings for the Full Catalog

Load the best checkpoint, then embed every image in the catalog once. These embeddings are
what powers retrieval at query time.


In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

full_ds = CatalogImageDataset(catalog, eval_transform)
full_loader = DataLoader(full_ds, batch_size=128, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

all_embeddings = np.zeros((len(catalog), EMBED_DIM), dtype=np.float32)

with torch.no_grad():
    for imgs, idxs in full_loader:
        imgs = imgs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            emb = model(imgs)
        all_embeddings[idxs.numpy()] = emb.cpu().numpy().astype(np.float32)

print("Embeddings shape:", all_embeddings.shape)

np.save(os.path.join(OUTPUT_DIR, "catalog_embeddings.npy"), all_embeddings)
catalog.to_csv(os.path.join(OUTPUT_DIR, "catalog_index.csv"), index=False)


## 8. Build the Similarity Search Index (FAISS)

Embeddings are L2-normalized, so inner product == cosine similarity. We use a flat
(exact) FAISS index — for 50K images this is fast; for much larger catalogs swap in
`IndexIVFFlat` or `IndexHNSWFlat`.


In [ ]:
try:
    import faiss
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "faiss-cpu", "--quiet"], check=True)
    import faiss

index = faiss.IndexFlatIP(EMBED_DIM)   # inner product on normalized vectors = cosine similarity
index.add(all_embeddings)
print("FAISS index size:", index.ntotal)

faiss.write_index(index, os.path.join(OUTPUT_DIR, "catalog.faiss"))


## 9. Query Function — Find Visually Similar Shoes

In [ ]:
def embed_image(pil_image_or_path):
    if isinstance(pil_image_or_path, (str, Path)):
        img = Image.open(pil_image_or_path).convert("RGB")
    else:
        img = pil_image_or_path.convert("RGB")
    tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        emb = model(tensor)
    return emb.cpu().numpy().astype(np.float32)


def find_similar_shoes(query_image_path, top_k=8, exclude_query_if_in_catalog=True):
    """
    Given a path to a query image (e.g. user upload), return the top_k most
    visually similar catalog rows + similarity scores.
    """
    query_emb = embed_image(query_image_path)
    search_k = top_k + 1 if exclude_query_if_in_catalog else top_k
    scores, idxs = index.search(query_emb, search_k)
    scores, idxs = scores[0], idxs[0]

    results = catalog.iloc[idxs].copy()
    results["similarity"] = scores

    if exclude_query_if_in_catalog:
        results = results[results["path"] != str(query_image_path)]

    return results.head(top_k).reset_index(drop=True)


def show_results(query_image_path, results):
    n = len(results) + 1
    plt.figure(figsize=(3 * n, 4))

    ax = plt.subplot(1, n, 1)
    ax.imshow(Image.open(query_image_path).convert("RGB"))
    ax.set_title("QUERY", fontsize=10, fontweight="bold")
    ax.axis("off")

    for i, row in results.iterrows():
        ax = plt.subplot(1, n, i + 2)
        ax.imshow(Image.open(row["path"]).convert("RGB"))
        ax.set_title(f"{row['brand']}\nsim={row['similarity']:.3f}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# ---- Demo: pick a random catalog image and treat it as a "user upload" ----
demo_query_path = catalog.sample(1, random_state=1)["path"].values[0]
results = find_similar_shoes(demo_query_path, top_k=6)
show_results(demo_query_path, results)
results


## 10. Evaluation — Precision@K

We don't have ground-truth "these are the same shoe" labels for arbitrary uploads, so we use
category/subcategory agreement between the query and its retrieved neighbors as a proxy
metric for retrieval quality. This is evaluated on held-out validation images only.


In [ ]:
def precision_at_k(df_eval, k=5, n_queries=300):
    sample = df_eval.sample(min(n_queries, len(df_eval)), random_state=SEED)
    hits = []
    for _, row in sample.iterrows():
        res = find_similar_shoes(row["path"], top_k=k)
        match = (res["subcategory"] == row["subcategory"]).mean()
        hits.append(match)
    return float(np.mean(hits))

p_at_5 = precision_at_k(val_df, k=5)
print(f"Precision@5 (subcategory match) on held-out val images: {p_at_5:.3f}")


## 11. Inference on a Brand-New Uploaded Image

This is the function to call in production: pass any image path (e.g. a photo uploaded by a
marketplace user, not necessarily from this catalog) and get back the most visually similar
catalog items.


In [ ]:
def visual_search(uploaded_image_path, top_k=8):
    """Production entry point."""
    results = find_similar_shoes(uploaded_image_path, top_k=top_k, exclude_query_if_in_catalog=False)
    show_results(uploaded_image_path, results)
    return results[["path", "category", "subcategory", "brand", "similarity"]]

# Example usage:
# visual_search("/kaggle/input/my-uploaded-shoe-photo/query.jpg", top_k=8)


## 12. Save Deployment Artifacts

Everything needed to serve this model elsewhere (e.g. behind an API):
- `shoe_embedding_model.pt` — model weights
- `catalog_embeddings.npy` — precomputed catalog embeddings
- `catalog_index.csv` — metadata (path, category, subcategory, brand) aligned to embeddings
- `catalog.faiss` — FAISS index for fast nearest-neighbor search


In [ ]:
print("Saved artifacts in", OUTPUT_DIR, ":")
for f in ["shoe_embedding_model.pt", "catalog_embeddings.npy", "catalog_index.csv", "catalog.faiss"]:
    p = os.path.join(OUTPUT_DIR, f)
    print(" -", f, "OK" if os.path.exists(p) else "MISSING")


### Next steps / extensions
- **Better supervision:** use the provided pairwise relative-attribute labels
  (`zappos-labels.mat`: open / pointy / sporty / comfort) with a ranking loss instead of, or
  alongside, the category-based triplets used here — this targets fine-grained visual
  attributes rather than just brand/category.
- **Hard-negative mining:** replace random negative sampling with semi-hard/hard negatives
  for a sharper embedding space.
- **Backbone upgrade:** swap ResNet50 for a stronger/lighter backbone (EfficientNet, CLIP
  image encoder, DINOv2) — a frozen CLIP encoder is a strong zero-shot baseline worth
  comparing against.
- **Scaling the index:** for catalogs beyond ~1M images, switch `IndexFlatIP` to
  `IndexIVFFlat` or `IndexHNSWFlat` for approximate but much faster search.
- **Serving:** wrap `visual_search()` in a FastAPI endpoint that accepts an uploaded image,
  runs `embed_image` + FAISS search, and returns product IDs/similarity scores as JSON.
